# Shared Camera (via iceoryx2)

`SharedCamera` gives every consumer (threads, processes, even programs written
in different languages) zero-copy access to a single physical camera through
shared memory. A detached publisher subprocess owns the device; subscribers
just call `read()` or `read_latest()` without worrying about who else is using
the camera, buffer management, or copy overhead.

Because the transport layer is [iceoryx2](https://github.com/eclipse-iceoryx/iceoryx2),
any language with iceoryx2 bindings (Rust, C, C++, Python, …) can subscribe to
the same camera stream.

**Why use SharedCamera:**

- **No exclusive-access headaches.** One publisher opens the camera; everyone
  else reads from shared memory. No "device busy" errors, no coordination code.
- **Zero-copy by default.** Frames live in a shared memory segment. Subscribers
  get a direct view, with no serialization or copies between processes.
- **Cross-language.** The SHM layout is language-agnostic. Python and Rust/C++
  can consume the same camera simultaneously.
- **Fully managed.** The publisher runs in a separate process and outlives any
  individual subscriber. Subscribers can connect and disconnect freely without
  reinitializing the camera or managing lifecycles. `SharedCamera` handles it all.

**Prerequisites:**

```bash
uv pip install physicalai[notebooks,capture,transport]
```

## 1. Discover cameras

In [ ]:
from physicalai.capture import CameraType, discover_all

results = discover_all()
for driver, devices in results.items():
    for dev in devices:
        serial = f"  serial={dev.hardware_id}" if dev.hardware_id else ""
        print(f"[{driver}] {dev.name or dev.device_id}{serial}")

if not any(results.values()):
    raise RuntimeError("No cameras found.")

## 2. Connect

Pick a camera type and device from the list above. `SharedCamera` auto-spawns
a publisher subprocess if none is running for this device.

In [ ]:
from physicalai.capture import SharedCamera

# --- Edit these to match your setup ---
CAMERA_TYPE = CameraType.UVC
DEVICE_KWARGS = {"device": 0}  # or {"serial_number": "123456"} for realsense/basler
# ---------------------------------------

cam = SharedCamera(CAMERA_TYPE, zero_copy=True, **DEVICE_KWARGS)
cam.connect()
print(f"Connected to {cam.device_id}")

## 3. Capture a frame

In [ ]:
import matplotlib.pyplot as plt

frame = cam.read_latest()
print(f"shape={frame.data.shape}  dtype={frame.data.dtype}  seq={frame.sequence}")

plt.figure(figsize=(8, 6))
plt.imshow(frame.data)
plt.title(f"Frame #{frame.sequence}")
plt.axis("off")
plt.show()

## 4. Measure latency

`frame.timestamp` is the publisher's `time.monotonic()` at capture time.
Both processes share the same system monotonic clock, so the difference
gives true capture-to-read latency.

In [ ]:
import time

NUM_SAMPLES = 100
latencies = []
for _ in range(NUM_SAMPLES):
    f = cam.read()
    latencies.append((time.monotonic() - f.timestamp) * 1000)

print(f"Samples: {NUM_SAMPLES}")
print(f"Mean:    {sum(latencies) / len(latencies):.2f} ms")
print(f"Min:     {min(latencies):.2f} ms")
print(f"Max:     {max(latencies):.2f} ms")

plt.figure(figsize=(8, 3))
plt.hist(latencies, bins=30, edgecolor="black")
plt.xlabel("Latency (ms)")
plt.ylabel("Count")
plt.title("Capture-to-read latency")
plt.tight_layout()
plt.show()

## 5. Multiple subscribers

Multiple `SharedCamera` instances can subscribe to the same publisher.
Each gets independent zero-copy access.

In [ ]:
cam2 = SharedCamera.from_publisher(cam.service_name, zero_copy=True)
cam2.connect()

f1 = cam.read_latest()
f2 = cam2.read_latest()
print(f"Subscriber 1: seq={f1.sequence}")
print(f"Subscriber 2: seq={f2.sequence}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(f1.data)
ax1.set_title(f"Sub 1  seq={f1.sequence}")
ax1.axis("off")
ax2.imshow(f2.data)
ax2.set_title(f"Sub 2  seq={f2.sequence}")
ax2.axis("off")
plt.tight_layout()
plt.show()

cam2.disconnect()

## 6. Reconnecting from another process

The publisher keeps running independently. Any new script or process can attach
to an already-running publisher by service name, with no camera re-initialization.

In [ ]:
# A second script / process can attach to the same publisher.
cam = SharedCamera.from_publisher("physicalai/camera/uvc/0/frame", zero_copy=True)
cam.connect()
frame = cam.read_latest()
print(f"Attached to existing publisher, seq={frame.sequence}  shape={frame.data.shape}")

## 7. Disconnect

The publisher self-terminates after all subscribers disconnect (configurable
`idle_timeout`, default 5 s).

In [ ]:
cam.disconnect()
print(f"Connected: {cam.is_connected}")